In [ ]:
import numpy as np
import trimesh
import rtree
import pyvista as pv
import plotly

In [ ]:

mesh_or_scene = trimesh.load("../data/tung_tung_sahur.glb")

if isinstance(mesh_or_scene, trimesh.Scene):
    mesh = trimesh.util.concatenate(
        tuple(mesh_or_scene.geometry.values())
    )
else:
    mesh = mesh_or_scene

original_mesh = mesh

In [ ]:
"""import plotly.graph_objects as go

v = original_mesh.vertices
f = original_mesh.faces

fig = go.Figure(data=[
    go.Mesh3d(
        x=v[:,0],
        y=v[:,1],
        z=v[:,2],
        i=f[:,0],
        j=f[:,1],
        k=f[:,2],

        color="saddlebrown",   # neutral clay color
        opacity=1.0,

        flatshading=False,  # smooth shading

        lighting=dict(
            ambient=0.15,   # base illumination
            diffuse=0.9,    # strong surface shading
            specular=0.8,   # shiny highlights
            roughness=0.2,  # smooth surface
            fresnel=0.2     # edge brightness
        ),

        lightposition=dict(
            x=100,
            y=200,
            z=300
        )
    )
])

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_visible=False,
        yaxis_visible=False,
        zaxis_visible=False
    ),
    width=900,
    height=800
)

fig.show()"""

In [ ]:
# crop out baseball bat
# compute triangle centroids
triangles = mesh.triangles
centroids = triangles.mean(axis=1)

# keep only triangles with x <= 0.4
mask = centroids[:,0] <= 0.4

# rebuild mesh
mesh = mesh.submesh([mask], append=True)

mesh

In [ ]:
#mesh.export("../data/tung_tung_sahur_cropped.glb")

In [ ]:
#mesh = trimesh.load("../data/tung_tung_sahur_cropped.glb")

In [ ]:
print("Vertices:", mesh.vertices.shape)
print("Faces:", mesh.faces.shape)

print("\nBounding box:")
print(mesh.bounds)

print("\nMesh center:")
print(mesh.centroid)

print("\nMesh extents (size in x,y,z):")
print(mesh.extents)

# visualizations

In [ ]:
# visualization with triangles mesh
import plotly.graph_objects as go
import numpy as np

v = mesh.vertices
f = mesh.faces

# --- surface ---
mesh_plot = go.Mesh3d(
    x=v[:,0],
    y=v[:,1],
    z=v[:,2],
    i=f[:,0],
    j=f[:,1],
    k=f[:,2],
    color="saddlebrown",
    opacity=0.6,
    flatshading=True
)

# --- triangle edges ---
edge_x = []
edge_y = []
edge_z = []

for tri in f:
    pts = v[tri]

    # close triangle loop
    for i in range(4):
        p = pts[i % 3]
        edge_x.append(p[0])
        edge_y.append(p[1])
        edge_z.append(p[2])
    edge_x.append(None)
    edge_y.append(None)
    edge_z.append(None)

wireframe = go.Scatter3d(
    x=edge_x,
    y=edge_y,
    z=edge_z,
    mode="lines",
    line=dict(color="blue", width=1)
)

fig = go.Figure(data=[mesh_plot, wireframe])

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    width=900,
    height=800
)

fig.show()

In [ ]:
import plotly.graph_objects as go

v = mesh.vertices
f = mesh.faces

fig = go.Figure(data=[
    go.Mesh3d(
        x=v[:,0],
        y=v[:,1],
        z=v[:,2],
        i=f[:,0],
        j=f[:,1],
        k=f[:,2],

        color="saddlebrown",   # neutral clay color
        opacity=1.0,

        flatshading=False,  # smooth shading

        lighting=dict(
            ambient=0.15,   # base illumination
            diffuse=0.9,    # strong surface shading
            specular=0.8,   # shiny highlights
            roughness=0.2,  # smooth surface
            fresnel=0.2     # edge brightness
        ),

        lightposition=dict(
            x=100,
            y=200,
            z=300
        )
    )
])

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_visible=False,
        yaxis_visible=False,
        zaxis_visible=False
    ),
    width=900,
    height=800
)

fig.show()

### calcualting distance to traingles centroid for probability calculations

In [ ]:
prox = trimesh.proximity.ProximityQuery(mesh)

In [ ]:
def distance_to_cow(point):
    point = np.array(point).reshape(1,3)
    return prox.signed_distance(point)[0]

In [ ]:
# test a few points
points = [
    [0,1,0],
    [0.5,1,0],
    [2,2,2]
]

for p in points:
    d = distance_to_cow(p)
    print(f"Point {p} -> distance {d}")

In [ ]:
mins, maxs = mesh.bounds

for _ in range(5):
    
    p = np.random.uniform(mins, maxs)
    
    d = distance_to_cow(p)
    
    print("point:", p, "distance:", d)

# MH algorithm

In [ ]:
def metropolis_sampler(n_samples=50000, sigma=0.05):

    mins, maxs = mesh.bounds
    
    # initialize point randomly in bounding box
    x = np.random.uniform(mins, maxs)
    
    samples = []
    
    d = distance_to_cow(x)
    
    for i in range(n_samples):
        
        # propose new point
        proposal = x + np.random.normal(0, sigma, 3)
        
        d_new = distance_to_cow(proposal)
        
        # MH acceptance ratio
        log_r = -100 * (abs(d_new) - abs(d))
        r = np.exp(min(0, log_r))

        if np.random.rand() < min(1, r):
            x = proposal
            d = d_new
        
        samples.append(x.copy())
    
    return np.array(samples)

In [ ]:
samples = metropolis_sampler(50000)

samples.shape

### visualize samples genereated

In [ ]:
import plotly.graph_objects as go
import numpy as np

# optional subsample
max_points = 50000
pts = samples if len(samples) <= max_points else samples[np.random.choice(len(samples), max_points, replace=False)]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=pts[:,0],
    y=pts[:,1],
    z=pts[:,2],
    mode="markers",
    marker=dict(
        size=2,
        color="red",
        opacity=0.7
    )
))

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    width=800,
    height=700
)

fig.show()

In [ ]:
import plotly.graph_objects as go

traj = samples[:20000]   # shorter path for clarity

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=traj[:,0],
    y=traj[:,1],
    z=traj[:,2],
    mode="lines+markers",
    line=dict(width=3, color="blue"),
    marker=dict(size=2, color="red")
))

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    width=800,
    height=700
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# optional: subsample samples if there are too many
max_points = 10000
pts = samples if len(samples) <= max_points else samples[np.random.choice(len(samples), max_points, replace=False)]

v = mesh.vertices
f = mesh.faces

fig = go.Figure()

# --- mesh (faint) ---
fig.add_trace(go.Mesh3d(
    x=v[:,0],
    y=v[:,1],
    z=v[:,2],
    i=f[:,0],
    j=f[:,1],
    k=f[:,2],

    color="#d8d8d8",
    opacity=0.18,
    flatshading=False,

    lighting=dict(
        ambient=0.15,
        diffuse=0.9,
        specular=0.8,
        roughness=0.3
    ),

    lightposition=dict(
        x=100,
        y=200,
        z=300
    )
))

# --- Monte Carlo samples ---
fig.add_trace(go.Scatter3d(
    x=pts[:,0],
    y=pts[:,1],
    z=pts[:,2],
    mode="markers",
    marker=dict(
        size=2,
        color="red",
        opacity=0.6
    )
))

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_visible=False,
        yaxis_visible=False,
        zaxis_visible=False
    ),
    width=900,
    height=800
)

fig.show()

#### animation

In [ ]:
import plotly.graph_objects as go
import numpy as np

v = mesh.vertices
f = mesh.faces

# limit trajectory for smoother animation
traj = samples[:20000]

step = 40   # frames step size

frames = []

for k in range(step, len(traj), step):

    frames.append(
        go.Frame(
            data=[
                go.Scatter3d(
                    x=traj[:k,0],
                    y=traj[:k,1],
                    z=traj[:k,2],
                    mode="lines+markers",
                    line=dict(color="red", width=3),
                    marker=dict(size=2, color="orange")
                )
            ]
        )
    )

fig = go.Figure(
    data=[

        # mesh
        go.Mesh3d(
            x=v[:,0],
            y=v[:,1],
            z=v[:,2],
            i=f[:,0],
            j=f[:,1],
            k=f[:,2],
            color="#d8d8d8",
            opacity=0.15,
            flatshading=False
        ),

        # starting trajectory
        go.Scatter3d(
            x=traj[:step,0],
            y=traj[:step,1],
            z=traj[:step,2],
            mode="lines+markers",
            line=dict(color="red", width=3),
            marker=dict(size=2, color="orange")
        )
    ],

    frames=frames
)

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_visible=False,
        yaxis_visible=False,
        zaxis_visible=False
    ),
    width=900,
    height=800,

    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 60}, "fromcurrent": True}]
            }
        ]
    }]
)

fig.show()